In [1]:
# %%
import sys
import os
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import balanced_accuracy_score, f1_score

import warnings
warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    module="transformers"
)

# TabPFN Import
try:
    from tabpfn import TabPFNClassifier
    TABPFN_AVAILABLE = True
except ImportError:
    TABPFN_AVAILABLE = False

# Setup Path
def find_project_root(name='Bambino', start=None):
    if start is None: start = os.getcwd()
    parent = start
    while True:
        if os.path.basename(parent) == name: return parent
        if os.path.dirname(parent) == parent: return None
        parent = os.path.dirname(parent)

PROJECT_ROOT = find_project_root()
if PROJECT_ROOT: sys.path.append(PROJECT_ROOT)

import config_moment as cfg
import moment_utils
import eval_utils
from DataUtils.BoaOpenFaceDataset import BoaOpenFaceDataset
from config import settings as global_settings


In [2]:
# %% --- 1. Load Data & Extract Embeddings ---
print(">>> Step 1: Loading Data & Extracting MOMENT Embeddings")

extractor = moment_utils.MomentFeatureExtractor()

# Load Datasets
train_path = global_settings.get_dataset_path(cfg.dataset_type, global_settings.training_filename)
val_path = global_settings.get_dataset_path(cfg.dataset_type, global_settings.validation_filename)
test_path = global_settings.get_dataset_path(cfg.dataset_type, global_settings.test_filename)

train_ds = BoaOpenFaceDataset.load_dataset(train_path)
val_ds = BoaOpenFaceDataset.load_dataset(val_path)
test_ds = BoaOpenFaceDataset.load_dataset(test_path)

# Extract (Embeddings + Labels + Metadata)
X_train_emb, y_train, meta_train = extractor.extract(train_ds)
X_val_emb,   y_val,   meta_val   = extractor.extract(val_ds)
X_test_emb,  y_test,  meta_test  = extractor.extract(test_ds)

print(f"Embedding Shape: {X_train_emb.shape}")

>>> Step 1: Loading Data & Extracting MOMENT Embeddings
Loading MOMENT (embedding) on cuda...


/home/phd2/Scrivania/CorsoVenvs/BambinoVenv/lib/python3.11/site-packages/momentfm/models/moment.py:174: UserWarning: Only reconstruction head is pre-trained. Classification and forecasting heads must be fine-tuned.
  warnings.warn("Only reconstruction head is pre-trained. Classification and forecasting heads must be fine-tuned.")


Extracting Embeddings (Batch Size 8)...


100%|██████████| 78/78 [00:54<00:00,  1.43it/s]


Extracting Embeddings (Batch Size 8)...


100%|██████████| 18/18 [00:11<00:00,  1.52it/s]


Extracting Embeddings (Batch Size 8)...


100%|██████████| 18/18 [00:11<00:00,  1.56it/s]

Embedding Shape: (619, 1024)


In [3]:
# %% --- 2. Feature Fusion (Embeddings + Age + Sex) ---
print(">>> Step 2: Fusing Metadata")

scaler = StandardScaler()
# Normalize Age (Train fit, Val/Test transform)
age_train = scaler.fit_transform(meta_train[['age']].values)
age_val   = scaler.transform(meta_val[['age']].values)
age_test  = scaler.transform(meta_test[['age']].values)

# Encode Sex (Already 0/1 usually, ensure float/int)
sex_train = meta_train[['sex']].values.astype(float)
sex_val   = meta_val[['sex']].values.astype(float)
sex_test  = meta_test[['sex']].values.astype(float)

def fuse_features(emb, age, sex):
    return np.hstack([emb, age, sex])

X_train_full = fuse_features(X_train_emb, age_train, sex_train)
X_val_full   = fuse_features(X_val_emb,   age_val,   sex_val)
X_test_full  = fuse_features(X_test_emb,  age_test,  sex_test)

>>> Step 2: Fusing Metadata


In [4]:
# %% --- 3. Classifier Selection & Training ---
print(f">>> Step 3: Training Classifier ({cfg.model_type})")

# create a dict to map models with numbers
model_type_dict = {
    1: 'embeddings+histgb',
    2: 'pca+tabpfn',
    3: 'tabpfn',
    4: 'embeddings+mlp',
    5: 'embeddings+logreg'
}

# set model_type variable here
model_choice = 5  # Change this number to select a different model
model_type = model_type_dict.get(model_choice, "embeddings+histgb")

if model_type is not None:
    print(f"Warning: model_type variable '{model_type}' defined here, not in config file. Using model_type defined here.")
else:
    print(f"Using model_type from config: {cfg.model_type}")
    model_type = cfg.model_type

# Output Setup
OUTPUT_DIR = Path(f"./results_{model_type}_{cfg.dataset_type}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

clf = None

if model_type == 'embeddings+histgb':
    # HistGradientBoosting (Robust, handles dense data well)
    clf = HistGradientBoostingClassifier(
        max_iter=300,
        learning_rate=0.05,
        max_depth=3,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=20,
        class_weight='balanced',
        random_state=cfg.seed
    )
    clf.fit(X_train_full, y_train)

    print("   -> HistGradientBoosting Training Complete.")


elif model_type == 'pca+tabpfn':
    if not TABPFN_AVAILABLE:
        raise ImportError("TabPFN not installed! Run `pip install tabpfn`")
    
    print("   -> Applying PCA before TabPFN (TabPFN limit ~100 feats)")
    # Pipeline: PCA -> TabPFN
    # Note: TabPFN is small-data specialized (N < 2000). If N > 2000, consider bagging.
    clf = Pipeline([
        ('pca', PCA(n_components=100)), # Reduce 1024 -> 100
        ('tabpfn', TabPFNClassifier(device='cuda', n_estimators=32))
    ])
    clf.fit(X_train_full, y_train)

    print("   -> PCA + TabPFN Training Complete.")


elif model_type == "tabpfn":
    if not TABPFN_AVAILABLE:
        raise ImportError("TabPFN not installed! Run `pip install tabpfn`")
    
    print("   -> Using TabPFN without PCA (Ensure feature count is manageable)")
    clf = TabPFNClassifier(device='cuda', n_estimators=32)
    clf.fit(X_train_full, y_train)

    print("   -> TabPFN Training Complete.")


elif model_type == 'embeddings+mlp':
    # Simple MLP
    clf = MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation='relu',
        solver='adam',
        alpha=0.001,
        batch_size=32,
        learning_rate='adaptive',
        max_iter=100,
        early_stopping=True,
        random_state=cfg.seed
    )
    clf.fit(X_train_full, y_train)

    print("   -> MLP Training Complete.")


elif model_type == 'embeddings+logreg': 
    from sklearn.linear_model import LogisticRegression
    import warnings
    from sklearn.exceptions import ConvergenceWarning
    warnings.filterwarnings("ignore", category=ConvergenceWarning, module="sklearn.linear_model._sag")

    # Logistic Regression
    clf = LogisticRegression(
        max_iter=300,
        class_weight='balanced',
        random_state=cfg.seed,
        penalty='l2',
        solver='lbfgs',
    )
    clf.fit(X_train_full, y_train)

    print("   -> Logistic Regression Training Complete.")


else:
    raise ValueError(f"Unknown model_type: {model_type}")

>>> Step 3: Training Classifier (embeddings+mlp)
   -> Logistic Regression Training Complete.


In [5]:
# %% --- 4. Evaluation ---
print(">>> Step 4: Evaluation")

# Optimize Threshold on Validation
probs_val = clf.predict_proba(X_val_full)[:, 1]
best_thr = 0.5
best_score = 0.0

for thr in np.linspace(0.1, 0.9, 50):
    preds = (probs_val >= thr).astype(int)
    score = balanced_accuracy_score(y_val, preds)
    if score > best_score:
        best_score = score
        best_thr = thr

print(f"Optimal Threshold (Val): {best_thr:.2f} (BalAcc: {best_score:.4f})")

# Final Test Prediction
probs_test = clf.predict_proba(X_test_full)[:, 1]
preds_test = (probs_test >= best_thr).astype(int)

# Use shared eval utility
eval_utils.evaluate_and_plot(
    y_true=y_test, 
    y_pred=preds_test, 
    y_probs=probs_test, 
    meta_df=meta_test, 
    output_dir=OUTPUT_DIR,
    model_name=f"MOMENT_{model_type}"
)

print("\n✅ Pipeline Completed Successfully.")

>>> Step 4: Evaluation
Optimal Threshold (Val): 0.49 (BalAcc: 0.5893)

--- MOMENT_embeddings+logreg Final Results ---
Balanced Accuracy: 0.5323
MCC: 0.0514
Brier Score: 0.2552

Classification Report:
              precision    recall  f1-score   support

           0     0.2174    0.5556    0.3125        27
           1     0.8235    0.5091    0.6292       110

    accuracy                         0.5182       137
   macro avg     0.5205    0.5323    0.4709       137
weighted avg     0.7041    0.5182    0.5668       137

✅ Plots saved to results_embeddings+logreg_normalized

✅ Pipeline Completed Successfully.


<Figure size 600x500 with 0 Axes>